In [1]:

import pandas as pd
import numpy as np
from datetime import datetime

df_raw = pd.read_csv('../data/netflix_titles.csv')
df_raw.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
print('Rows, Columns:', df_raw.shape)
print('\nColumn dtypes:')
print(df_raw.dtypes)
df_raw.isnull().sum()  # should be all zeros

Rows, Columns: (8790, 10)

Column dtypes:
show_id           str
type              str
title             str
director          str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
dtype: object


show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [3]:
# Clean `duration` – separate minutes vs seasons
def parse_duration(val):
    if pd.isna(val):
        return np.nan
    # Movies: "90 min"  → 90
    if 'min' in val.lower():
        return int(val.lower().replace('min','').strip())
    # TV Shows: "2 Seasons" → 2
    if 'season' in val.lower():
        return int(val.lower().replace('seasons','').replace('season','').strip())
    return np.nan

df = df_raw.copy()
df['duration_num'] = df['duration'].apply(parse_duration)
df[['duration', 'duration_num']].head()

,duration,duration_num
0,90 min,90
1,1 Season,1
2,1 Season,1
3,91 min,91
4,125 min,125


In [4]:
#  Convert `date_added` to a proper datetime object
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
# Extract useful parts for later analysis
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df[['date_added','year_added','month_added']].head()

,date_added,year_added,month_added
0,2021-09-25,2021,9
1,2021-09-24,2021,9
2,2021-09-24,2021,9
3,2021-09-22,2021,9
4,2021-09-24,2021,9


In [5]:
# Standardise text columns (lower‑case, strip whitespace)
text_cols = ['title','director','country','rating','listed_in']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_num,year_added,month_added
0,s1,Movie,dick johnson is dead,kirsten johnson,united states,2021-09-25,2020,pg-13,90 min,documentaries,90,2021,9
1,s3,TV Show,ganglands,julien leclercq,france,2021-09-24,2021,tv-ma,1 Season,"crime tv shows, international tv shows, tv act...",1,2021,9
2,s6,TV Show,midnight mass,mike flanagan,united states,2021-09-24,2021,tv-ma,1 Season,"tv dramas, tv horror, tv mysteries",1,2021,9
3,s14,Movie,confessions of an invisible girl,bruno garotti,brazil,2021-09-22,2021,tv-pg,91 min,"children & family movies, comedies",91,2021,9
4,s8,Movie,sankofa,haile gerima,united states,2021-09-24,1993,tv-ma,125 min,"dramas, independent movies, international movies",125,2021,9


In [6]:
#  Replace the placeholder "Not Given" with NaN – easier for imputation
df.replace('not given', np.nan, inplace=True)
df.isna().sum()  # now we can see truly missing values

show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
duration_num       0
year_added         0
month_added        0
dtype: int64

In [7]:
#  Save the clean version for later notebooks
df.to_csv('../data/netflix_clean.csv', index=False)
print('Saved cleaned data to ../data/netflix_clean.csv')

Saved cleaned data to ../data/netflix_clean.csv
